In [11]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')  # NOTA: due punti, non 'memory:'
cursor = conn.cursor()

# DATI DI PRODUZIONE MINERARIA (dati sporchi)
data_produzione = {
    'ID_Mese': ['GEN23', 'feb23', 'MAR23', 'apr23', 'MAG23', 'giu23', 'LUG23', 'ago23', 'SET23', 'ott23', 'NOV23', 'dic23'],
    'Data': ['2023-01-01', '01/02/2023', '2023.03.01', '01-04-2023', '2023/05/01', '2023-06-01', '01/07/2023', '2023.08.01', '01-09-2023', '2023/10/01', '2023-11-01', '31-12-2023'],
    'Sito_estrazione': ['Nord', 'nord', 'SUD', 'Sud', 'Est', 'EST', 'ovest', 'Ovest', 'Nord', 'SUD', 'est', 'Nord'],
    'Minerali_estratti': ['Oro,Argento', 'oro,argento', 'Rame,Oro', 'Oro', 'Argento,Rame', 'Oro,Argento,Rame', 'Rame', 'Oro', 'Argento', 'Oro,Rame', 'Argento', 'Oro,Argento'],
    'Tonni_estratte': ['1250', '1350', '1100', '950', '880', '1020', '980', '1150', '1250', '1080', '1120', '1300'],
    'Oro_grammi': ['2.5', '2.8', '1.8', '2.1', '0', '2.2', '0', '2.4', '0', '2.0', '0', '2.3'],
    'Argento_grammi': ['15', '18', '0', '12', '8.5', '20', '0', '0', '25', '0', '15', '22'],
    'Rame_kg': ['120', '130', '150', '0', '90', '180', '200', '0', '0', '160', '0', '140'],
    'Ore_lavorate': ['720', '700', '680', '650', '620', '690', '640', '710', '680', '660', '670', '700'],
    'Note': ['Produzione regolare', 'Ritardo consegne', 'Manutenzione impianti', 'Nuovo fronte', 'Messa in sicurezza', 'Picco produttivo', 'Guasto macchinari', 'Produzione regolare', 'Sciopero', 'Record estrazione', 'Ferma per inventario', 'Fine anno']
}

# Carica in SQLite
df_produzione = pd.DataFrame(data_produzione)
df_produzione.to_sql('Produzione_Grezza', conn, index=False, if_exists='replace')

print("=== DATI GREZZI PRODUZIONE MINERARIA ===")
print(df_produzione)

=== DATI GREZZI PRODUZIONE MINERARIA ===
   ID_Mese        Data Sito_estrazione Minerali_estratti Tonni_estratte  \
0    GEN23  2023-01-01            Nord       Oro,Argento           1250   
1    feb23  01/02/2023            nord       oro,argento           1350   
2    MAR23  2023.03.01             SUD          Rame,Oro           1100   
3    apr23  01-04-2023             Sud               Oro            950   
4    MAG23  2023/05/01             Est      Argento,Rame            880   
5    giu23  2023-06-01             EST  Oro,Argento,Rame           1020   
6    LUG23  01/07/2023           ovest              Rame            980   
7    ago23  2023.08.01           Ovest               Oro           1150   
8    SET23  01-09-2023            Nord           Argento           1250   
9    ott23  2023/10/01             SUD          Oro,Rame           1080   
10   NOV23  2023-11-01             est           Argento           1120   
11   dic23  31-12-2023            Nord       Oro,Argento   

In [12]:
# Query di pulizia
q = """
SELECT 
 -- Pulizia ID_Mese (-> gen23')
UPPER(TRIM(ID_mese)) AS id_mese,

 -- Pulizia Data (-> formato unico YYYY-MM-DD)
CASE
WHEN Data LIKE '__/__/____' THEN 
    SUBSTR(Data, 7, 4) || '-' ||
    SUBSTR(Data, 4, 2) || '-' ||
    SUBSTR(Data, 1, 2)
WHEN Data LIKE '__-__-____' THEN 
    SUBSTR(Data, 7, 4) || '-' ||
    SUBSTR(Data, 4, 2) || '-' ||
    SUBSTR(Data, 1, 2)
WHEN Data LIKE '%.%' THEN 
    REPLACE(Data, '.', '-')
WHEN Data LIKE '%/%' THEN 
    REPLACE(Data, '/', '-')
ELSE Data
END AS data,

 -- Sito_estrazione (-> NORD)
UPPER(TRIM(Sito_estrazione)) AS sito_estrazione,

 -- Tonni_estratte (-> 1250.0)
CAST(TRIM(Tonni_estratte) AS FLOAT) AS ton_estratte,

 -- Oro_grammi (-> 2.5)
CAST(TRIM(Oro_grammi) AS FLOAT) AS oro_g,

 -- Argento_grammi (-> 15.0)
CAST(TRIM(Argento_grammi) AS FLOAT) AS argento_g,

 -- Rame_kg (-> 120.0)
CAST(TRIM(Rame_kg) AS FLOAT) AS rame_Kg,

 -- Ore_lavorate (-> 720.0)
CAST(TRIM(Ore_lavorate) AS FLOAT) AS ore_lavorative,

Note
FROM Produzione_Grezza;
"""

# Creiamo la tabella temporanea con i dati puliti (usa cursor dalla cella precedente)
cursor.execute("CREATE TEMPORARY TABLE Campioni_puliti_temp AS " + q)

# Verifichiamo il contenuto
print("\nProduzione_Grezza --> Puliti!")
df_verifica = pd.read_sql_query("SELECT * FROM Campioni_puliti_temp", conn)
print(df_verifica)


Produzione_Grezza --> Puliti!
   id_mese        data sito_estrazione  ton_estratte  oro_g  argento_g  \
0    GEN23  2023-01-01            NORD        1250.0    2.5       15.0   
1    FEB23  2023-02-01            NORD        1350.0    2.8       18.0   
2    MAR23  2023-03-01             SUD        1100.0    1.8        0.0   
3    APR23  2023-04-01             SUD         950.0    2.1       12.0   
4    MAG23  2023-05-01             EST         880.0    0.0        8.5   
5    GIU23  2023-06-01             EST        1020.0    2.2       20.0   
6    LUG23  2023-07-01           OVEST         980.0    0.0        0.0   
7    AGO23  2023-08-01           OVEST        1150.0    2.4        0.0   
8    SET23  2023-09-01            NORD        1250.0    0.0       25.0   
9    OTT23  2023-10-01             SUD        1080.0    2.0        0.0   
10   NOV23  2023-11-01             EST        1120.0    0.0       15.0   
11   DIC23  2023-12-31            NORD        1300.0    2.3       22.0   

    ra

## 📊 Calcolo parametri statistici fondamentali:

### Dopo la pulizia, calcola per Oro_grammi:

Misure di centro:
* Media
* Mediana
* Moda

Misure di spread:
* Range (max - min)
* Varianza
* Deviazione standard
* Coefficiente di variazione (CV = std/media)

Per ogni sito:
* Media Oro per sito
* Media Argento per sito
* Media Rame per sito
* Produttività media (Tonni_estratte / Ore_lavorate)